**Re-define virality with new data as a threshold of reposts on the post x amount of hours after creation.**

In [3]:
import pandas as pd
import numpy as np
from kneed import KneeLocator
import duckdb
import json
import tarfile
import matplotlib.pyplot as plt
from pathlib import Path
import sys


In [ ]:
BASE_PATH = Path.cwd().parent
sys.path.append(str(BASE_PATH))
root_posts = pd.read_parquet(BASE_PATH/"datasets/bluesky_cascade/root_posts.parquet")
print(root_posts.shape)

In [ ]:
# Compare distributions of reposts after x hours
diff_hours = [3,4,6,12,24]
repost_dist_after_each_num_hours = {}

for hours in diff_hours:
    repost_dist_after_x_hours = (
    root_posts[f"repost_{hours}h"]
    .value_counts()
    .sort_index()
    .reset_index()
)
    repost_dist_after_each_num_hours[hours] = repost_dist_after_x_hours

# Plot
fig, ax = plt.subplots(figsize=(20, 6))

ax.violinplot(
    [repost_dist_after_each_num_hours[h] for h in diff_hours],
    positions=range(1,25),
    showmedians=True
)

ax.set_xlabel("Hours after post creation")
ax.set_ylabel("Repost count")
ax.set_title("Distribution of reposts received within X hours of creation")
ax.set_xticks(range(1, 25, 2))
plt.tight_layout()
plt.show()

Analyse the knees for each number of hours

In [ ]:
knee_results = []

for hours in diff_hours:
    try:
        kneedle = KneeLocator(
            repost_dist_after_each_num_hours[hours]["repost_count"],
            repost_dist_after_each_num_hours[hours]["post_count"],
            curve="convex",
            direction="decreasing"
        )
        knee = kneedle.knee
    except Exception as e:
        knee = None

    knee_results.append({"hours": hours, "knee": knee})
    print(f"Hour {hours:2d}: knee at {knee} reposts")

knee_df = pd.DataFrame(knee_results)

# Plot knee over time
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(knee_df["hours"], knee_df["knee"], marker="o")
ax.set_xlabel("Hours after post creation")
ax.set_ylabel("Repost threshold (knee)")
ax.set_title("Virality threshold (knee of repost distribution) over time")
ax.set_xticks(range(1, 25, 2))
plt.tight_layout()
plt.show()

Analyse percentiles for each time period

In [ ]:
percentile_values = [0.9, 0.95, 0.98, 0.99, 0.999, 0.9999, 1]
percentile_results = []

for hours in diff_hours:
    row = {"hours": hours}
    counts = repost_dist_after_each_num_hours[hours]["repost_count"]
    for p in percentile_values:
        row[f"p{p}"] = counts.quantile(p)
    percentile_results.append(row)
    print(f"Hour {hours}: " + ", ".join(f"p{p}={row[f'p{p}']:.1f}" for p in percentile_values))

percentile_df = pd.DataFrame(percentile_results)

fig, ax = plt.subplots(figsize=(12, 5))
for p in percentile_values:
    ax.plot(percentile_df["hours"], percentile_df[f"p{p}"], marker="o", label=f"p{p}")
ax.set_xlabel("Hours after post creation")
ax.set_ylabel("Repost count threshold")
ax.set_title("Repost distribution percentiles over time")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
chosen_time_period = #FILL IN

**Define virality based on TI and S z-scores**

In [ ]:
eps = 1 # Laplace smoothing (shift repost_count and follower_count + 1) makes more sense for logs and has intuitive/interpretable meaning

TI_mean_sd = duckdb.query(f"""
        SELECT
            AVG(LN(like_count + reply_count + repost_count + {eps})) as TI_mean,
            STDDEV(LN(like_count + reply_count + repost_count + {eps})) as TI_std
        FROM root_posts as p
        WHERE p.langs LIKE '%en%'
    """).df()

In [ ]:
print(TI_mean_sd)

In [ ]:
followers = pd.read
# Precompute follower counts once
follower_counts = duckdb.query("""
    SELECT followed_id, COUNT(*) as follower_count
    FROM followers
    GROUP BY followed_id
""").df()

# Register as a duckdb table so it can be joined
duckdb.register("follower_counts", follower_counts)

In [ ]:
S_mean_sd = duckdb.query(f"""
    SELECT
        AVG(LN((p.repost_count + {eps} / (NULLIF(follower_counts.follower_count, 0) + {eps})) + {eps})) as S_mean,
        STDDEV(LN((p.repost_count + {eps}/ (NULLIF(follower_counts.follower_count, 0) + {eps})) + {eps})) as S_std
    FROM root_posts as p
    LEFT JOIN follower_counts ON p.user_id = follower_counts.followed_id
    WHERE p.langs LIKE '%en%'
""").df()

In [ ]:
print(S_mean_sd)

**Compute z-scores and get overall virality summary stats**

In [ ]:
TI_mean = TI_mean_sd["TI_mean"][0]
TI_std = TI_mean_sd["TI_std"][0]
S_mean = S_mean_sd["S_mean"][0]
S_std = S_mean_sd["S_std"][0]

threshold_reposts_h = # FILL IN

# Accumulators for TI/S z-score definition
total = 0
high_S = 0
high_TI = 0
high_both_TI_S = 0

# Accumulators for repost/speed definition
high_speed = 0
high_both_speed_TI_S = 0

# Step 1: Write enriched file with z-scores directly to Drive
duckdb.query(f"""
    COPY (
        SELECT
            p.*,
            (LN(like_count + reply_count + repost_count + {eps}) - {TI_mean}) / {TI_std} as TI_z,
            (LN((p.repost_count + {eps}/ (NULLIF(follower_counts.follower_count, 0) + {eps})) + {eps}) - {S_mean}) / {S_std} as S_z
        FROM root_posts as p
        LEFT JOIN follower_counts ON p.user_id = follower_counts.followed_id
        WHERE p.langs LIKE '%en%'
    ) TO '{BASE_PATH}/datasets/bluesky_cascade/root_posts_with_virality_metrics.parquet' (FORMAT PARQUET)
""")

# Step 2: Compute all metrics from the enriched file
counts = duckdb.query(f"""
    SELECT
        COUNT(*) as total,
        SUM(CASE WHEN S_z >= 3 THEN 1 ELSE 0 END) as high_S,
        SUM(CASE WHEN TI_z >= 3 THEN 1 ELSE 0 END) as high_TI,
        SUM(CASE WHEN S_z >= 3 AND TI_z >= 3 THEN 1 ELSE 0 END) as high_both_TI_S,
        SUM(CASE WHEN reposts_{chosen_time_period}h >= {threshold_reposts_h} THEN 1 ELSE 0 END) as high_speed,
        SUM(CASE WHEN S_z >= 3 AND TI_z >= 3 AND reposts_{chosen_time_period}h >= {threshold_reposts_h} THEN 1 ELSE 0 END) as high_both_speed_TI_S
    FROM root_posts
""").fetchone()

total                = counts[0]
high_S               = counts[1]
high_TI              = counts[2]
high_both_TI_S       = counts[3]
high_speed           = counts[4]
high_both_speed_TI_S = counts[5]

# Final proportions
overall_metrics = {
    "prop_high_S":                high_S / total,
    "prop_high_TI":               high_TI / total,
    "prop_high_TI_given_high_S":  high_both_TI_S / high_S if high_S > 0 else None,
    "prop_high_S_given_high_TI":  high_both_TI_S / high_TI if high_TI > 0 else None,
    "prop_high_speed":            high_speed / total,
    "prop_high_speed_given_TI_S": high_both_speed_TI_S / high_both_TI_S if high_both_TI_S > 0 else None,
    "prop_TI_S_given_high_speed": high_both_speed_TI_S / high_speed if high_speed > 0 else None,
}
print(overall_metrics)

Compute z-scores and mew metrics